# ACT training walkthrough
This notebook is a thin client of the same verified production APIs and CLI used outside Jupyter. It contains no training loop or preprocessing logic.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

from act_lab.application.training import load_act_config, verify_dataset

repo_root = Path.cwd()
if not (repo_root / "configs").exists():
    repo_root = repo_root.parent
dataset_path = Path(
    os.environ.get("ACT_LAB_DATASET", repo_root / "data/lerobot/pick-place")
)
run_path = Path(os.environ.get("ACT_LAB_RUN", repo_root / "runs/training/act-teaching"))
config_path = repo_root / "configs/training/act.toml"
config = load_act_config(config_path)
print(config)
if dataset_path.exists():
    print(verify_dataset(dataset_path))
else:
    print(f"Set ACT_LAB_DATASET to inspect a converted dataset: {dataset_path}")

Set `ACT_LAB_LAUNCH=1` only when you intend to start the production CLI. Device selection is always explicit.

In [ ]:
if os.environ.get("ACT_LAB_LAUNCH") == "1":
    subprocess.run(
        [
            "act-lab",
            "train",
            "act",
            "--dataset",
            str(dataset_path),
            "--output",
            str(run_path),
            "--device",
            os.environ.get("ACT_LAB_DEVICE", "cpu"),
            "--config",
            str(config_path),
        ],
        check=True,
    )

In [ ]:
metrics_path = run_path / "metrics.jsonl"
if metrics_path.exists():
    import matplotlib.pyplot as plt

    metrics = [
        json.loads(line) for line in metrics_path.read_text().splitlines() if line
    ]
    plt.plot([m["step"] for m in metrics], [m["loss"] for m in metrics])
    plt.xlabel("optimizer step")
    plt.ylabel("training loss")
    plt.grid(True)
else:
    print(f"No metrics yet: {metrics_path}")